# Paperless-ngx ML Data Platform — Chameleon Setup

Run this notebook in the **Chameleon Jupyter environment** to provision a VM and bring up the data platform.

**What this does:**
- Provisions an `m1.xlarge` VM on KVM@TACC
- Associates a floating IP
- Opens required security groups
- Installs Docker on the VM
- Clones the repo and runs `make up`

## Step 1 — Select project and site

In [ ]:
from chi import server, context, lease, network
import chi, os, datetime

context.version = "1.0"
context.choose_project()
context.choose_site(default="KVM@TACC")
username = os.getenv("USER")
print(f"Username: {username}")

## Step 2 — Reserve VM (8 hours)

In [ ]:
l = lease.Lease(
    f"lease-paperless-{username}",
    duration=datetime.timedelta(hours=8)
)
l.add_flavor_reservation(id=chi.server.get_flavor_id("m1.xlarge"), amount=1)
l.submit(idempotent=True)
l.show()

## Step 3 — Launch VM instance

In [ ]:
s = server.Server(
    f"node-paperless-{username}",
    image_name="CC-Ubuntu24.04",
    flavor_name=l.get_reserved_flavors()[0].name,
)
s.submit(idempotent=True)

## Step 4 — Assign floating IP

In [ ]:
s.associate_floating_ip()
s.refresh()
s.show(type="widget")

## Step 5 — Open security groups

In [ ]:
security_groups = [
    {"name": "allow-ssh",  "port": 22,   "description": "SSH"},
    {"name": "allow-5050", "port": 5050, "description": "Adminer UI"},
    {"name": "allow-9001", "port": 9001, "description": "MinIO Console"},
    {"name": "allow-8090", "port": 8090, "description": "Redpanda Console"},
    {"name": "allow-6333", "port": 6333, "description": "Qdrant"},
    {"name": "allow-8080", "port": 8080, "description": "Airflow UI"},
    {"name": "allow-8000", "port": 8000, "description": "Hypothytical API endpoints"},
]

for sg in security_groups:
    secgroup = network.SecurityGroup({"name": sg["name"], "description": sg["description"]})
    secgroup.add_rule(direction="ingress", protocol="tcp", port=sg["port"])
    secgroup.submit(idempotent=True)
    s.add_security_group(sg["name"])

print(f"Security groups applied: {[sg['name'] for sg in security_groups]}")

## Step 6 — Install Docker

In [ ]:
s.refresh()
s.check_connectivity()
s.execute("curl -sSL https://get.docker.com/ | sudo sh")
s.execute("sudo groupadd -f docker; sudo usermod -aG docker $USER")
print("Docker installed.")

## Step 7 — Clone repo and bring up the platform

**Edit the `REPO_URL` below to your actual GitHub repo URL before running.**

In [ ]:
REPO_URL = "https://github.com/REDES01/paperless_data.git"  # <-- edit this

s.execute(f"git clone {REPO_URL} ~/paperless_data")
s.execute("cd ~/paperless_data && sg docker -c 'docker compose -f docker/docker-compose.yaml up -d'")
print("Platform started.")

## Step 8 — Print access URLs

In [ ]:
s.refresh()
# Get floating IP from addresses
addresses = s.addresses
floating_ip = None
for net, addrs in addresses.items():
    for addr in addrs:
        if addr.get("OS-EXT-IPS:type") == "floating":
            floating_ip = addr["addr"]

print(f"Floating IP: {floating_ip}")
print(f"")
print(f"  MinIO Console   ->  http://{floating_ip}:9001  (admin / paperless_minio)")
print(f"  Adminer (PG)    ->  http://{floating_ip}:5050  (user / paperless_postgres)")
print(f"  Redpanda UI     ->  http://{floating_ip}:8090")
print(f"  Qdrant          ->  http://{floating_ip}:6333/dashboard")
print(f"")
print(f"  SSH: ssh -i ~/.ssh/id_rsa_chameleon cc@{floating_ip}")